In [6]:
import pytesseract
from PIL import Image
from pathlib import Path

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

img = Image.open(Path.home() / "Documents" / "tea-payroll" / "receipts" / "sample.jpg")
img = img.rotate(180)

text = pytesseract.image_to_string(img)
print(text)

Receipt of lea Leaves

ein ii hs is ams wisn sa i i aa tn wl ea a es >
‘one Sais tu Gb tan Ws 0b ne ls ae a sms a is Ah aa ome wl wm, PSP

KTDA: NOIMA(39)

VERSION ; 1.0.56
eter Tet Te eteeere tet eet e tot
Centre ‘NEW KLARUHTU( 1)
Receipt No :330039762
Plucked 9626-01-14 13:56:47.224
Date 9028-01-14 13:56:47,224
Farmer ‘JUSTIN KAHUTHU( 10001 )
Tare Weight :0.50KG
No of Bags :2

-

Bag No Act Wt(Kg) Net Wt(Kg)
EERE AAA AA PATA
1 12 .3KG 11.8KG
2 12 .5KG 12 .6KG

Tea Weight 9 :23.8KG

Total Receipt For Rash :

Net Wt(Kg) 864.2 KG

cums KINYUA WANJOHT) (CHARLES KINYUA WANJL




In [8]:
import re
from datetime import datetime

# Known OCR mis-reads specific to this receipt printer font
OCR_CORRECTIONS = {
    "NOIMA": "NDIMA",
    "KLARUHTU": "KIARUHIU",
    "KLARUHIU": "KIARUHIU",
}

def correct_ocr(text):
    for wrong, right in OCR_CORRECTIONS.items():
        text = text.replace(wrong, right)
    return text

def parse_ktda_receipt(raw_text):
    # Apply known corrections first
    text = correct_ocr(raw_text)

    def grab(pattern, flags=0):
        m = re.search(pattern, text, flags)
        return m.group(1).strip() if m else None

    # \W+ means "any non-word characters" — handles : ' space or any combo
    centre  = re.search(r"Centre\W+([A-Z][A-Z\s]+?)\(\s*(\d+)\s*\)", text)
    factory = re.search(r"KTDA\W+([A-Z]+)\(\s*(\d+)\s*\)", text)
    farmer  = re.search(r"Farmer\W+([A-Z][A-Z\s]+?)\(\s*(\d+)\s*\)", text)

    # Date: \W+ handles missing or garbled colon, fix year OCR errors
    date_raw = grab(r"Plucked\W+(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})")
    if date_raw:
        # Fix year — OCR often misreads 2 as 9 at start of year
        date_raw = re.sub(r"^\d{4}", "2026", date_raw)

    data = {
        "receipt_no":     grab(r"Receipt\W+No\W+(\d+)"),
        "centre_name":    centre.group(1).strip() if centre else None,
        "centre_code":    centre.group(2).strip() if centre else None,
        "factory_name":   factory.group(1).strip() if factory else None,
        "factory_code":   factory.group(2).strip() if factory else None,
        "farmer_name":    farmer.group(1).strip() if farmer else None,
        "farmer_ktda_id": farmer.group(2).strip() if farmer else None,
        "plucked_at":     datetime.strptime(date_raw, "%Y-%m-%d %H:%M:%S") if date_raw else None,
        "tare_weight_kg": float(grab(r"Tare\W+Weight\W+([\d.]+)") or 0),
        "bag_count":      int(grab(r"No\W+of\W+Bags\W+(\d+)") or 0),
        "net_total_kg":   float(grab(r"Tea\W+Weight[^:]*\W+([\d.]+)") or 0),
        "mtd_kg_printed": float(grab(r"Net\s*Wt\(Kg\)\s+([\d.]+)\s*KG\s*$",
                                     re.MULTILINE) or 0),
    }

    # Per-bag rows — strip spaces OCR inserts into numbers like "12 .3KG"
    bag_re = re.compile(
        r"^\s*(\d+)\s+([\d\s.]+?)\s*KG\s+([\d\s.]+?)\s*KG",
        re.MULTILINE
    )
    data["bags"] = []
    for m in bag_re.finditer(text):
        act = float(m.group(2).replace(" ", ""))
        net = float(m.group(3).replace(" ", ""))
        data["bags"].append({
            "bag_no":    int(m.group(1)),
            "act_wt_kg": act,
            "net_wt_kg": net
        })

    bag_sum = sum(b["net_wt_kg"] for b in data["bags"])
    data["audit_ok"] = abs(bag_sum - data["net_total_kg"]) < 0.1

    return data

parsed = parse_ktda_receipt(text)
for key, value in parsed.items():
    print(f"{key:20} {value}")

receipt_no           330039762
centre_name          NEW KIARUHIU
centre_code          1
factory_name         NDIMA
factory_code         39
farmer_name          JUSTIN KAHUTHU
farmer_ktda_id       10001
plucked_at           2026-01-14 13:56:47
tare_weight_kg       0.5
bag_count            2
net_total_kg         23.8
mtd_kg_printed       864.2
bags                 [{'bag_no': 1, 'act_wt_kg': 12.3, 'net_wt_kg': 11.8}, {'bag_no': 2, 'act_wt_kg': 12.5, 'net_wt_kg': 12.6}]
audit_ok             False


In [9]:
# ── Manual corrections ─────────────────────────────────
# Fix bag 2 net weight (OCR read 12.6, actual on receipt is 12.0)
parsed["bags"][1]["net_wt_kg"] = 12.0

# Recalculate audit after correction
bag_sum = sum(b["net_wt_kg"] for b in parsed["bags"])
parsed["audit_ok"] = abs(bag_sum - parsed["net_total_kg"]) < 0.1

print("After correction:")
print("Bag weights:", [b["net_wt_kg"] for b in parsed["bags"]])
print("Bag sum:    ", bag_sum)
print("Tea Weight: ", parsed["net_total_kg"])
print("audit_ok:   ", parsed["audit_ok"])

After correction:
Bag weights: [11.8, 12.0]
Bag sum:     23.8
Tea Weight:  23.8
audit_ok:    True


In [10]:
for key, value in parsed.items():
    print(f"{key:20} {value}")

receipt_no           330039762
centre_name          NEW KIARUHIU
centre_code          1
factory_name         NDIMA
factory_code         39
farmer_name          JUSTIN KAHUTHU
farmer_ktda_id       10001
plucked_at           2026-01-14 13:56:47
tare_weight_kg       0.5
bag_count            2
net_total_kg         23.8
mtd_kg_printed       864.2
bags                 [{'bag_no': 1, 'act_wt_kg': 12.3, 'net_wt_kg': 11.8}, {'bag_no': 2, 'act_wt_kg': 12.5, 'net_wt_kg': 12.0}]
audit_ok             True


In [12]:
import os
import psycopg2
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path

# ── Reconnect ───────────────────────────────────────────
env_path = Path.home() / "Documents" / "tea-payroll" / ".env"
load_dotenv(env_path, override=True)

conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    dbname=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
)
print("Reconnected to database")

cur = conn.cursor()

# Get the rate that was active on the pluck date
cur.execute("""
    SELECT rate_kes_per_kg FROM rates
    WHERE effective_date <= %s
    ORDER BY effective_date DESC LIMIT 1
""", (parsed["plucked_at"],))

rate_row = cur.fetchone()
if not rate_row:
    print("ERROR: No rate found for this date. Check your rates table.")
else:
    rate = rate_row[0]
    print(f"Rate found: {rate} KES/kg")

    # Save the receipt header
    cur.execute("""
        INSERT INTO receipts (
            receipt_no, centre_name, centre_code,
            factory_name, factory_code,
            farmer_ktda_id, farmer_name,
            plucked_at, tare_weight_kg, bag_count,
            net_total_kg, mtd_kg_printed,
            photo_path, ocr_raw_text, confirmed_by_human
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s, TRUE)
        RETURNING receipt_id
    """, (
        parsed["receipt_no"],
        parsed["centre_name"],  parsed["centre_code"],
        parsed["factory_name"], parsed["factory_code"],
        parsed["farmer_ktda_id"], parsed["farmer_name"],
        parsed["plucked_at"],
        parsed["tare_weight_kg"], parsed["bag_count"],
        parsed["net_total_kg"],   parsed["mtd_kg_printed"],
        "receipts/sample.jpg", text
    ))

    receipt_id = cur.fetchone()[0]
    print(f"Receipt saved with ID: {receipt_id}")

    # Allocate each bag to a worker
    # Change these to match who actually plucked each bag
    bag_owners = {
        1: "W-0042",   # Mary Wanjiru plucked bag 1
        2: "W-0017",   # John Kamau plucked bag 2
    }

    for bag in parsed["bags"]:
        worker = bag_owners.get(bag["bag_no"])
        cur.execute("""
            INSERT INTO receipt_bags (
                receipt_id, bag_no,
                act_wt_kg, net_wt_kg,
                worker_id, rate_kes_per_kg
            )
            VALUES (%s,%s,%s,%s,%s,%s)
        """, (
            receipt_id,
            bag["bag_no"],
            bag["act_wt_kg"],
            bag["net_wt_kg"],
            worker,
            rate
        ))
        print(f"  Bag {bag['bag_no']}: {bag['net_wt_kg']} kg → {worker} @ {rate} KES/kg = {bag['net_wt_kg'] * float(rate):.2f} KES")

    conn.commit()
    print()
    print("All saved. Verifying from database:")
    df = pd.read_sql("SELECT * FROM v_payments_due", conn)
    print(df.to_string())

Reconnected to database
Rate found: 15.00 KES/kg
Receipt saved with ID: 1
  Bag 1: 11.8 kg → W-0042 @ 15.00 KES/kg = 177.00 KES
  Bag 2: 12.0 kg → W-0017 @ 15.00 KES/kg = 180.00 KES

All saved. Verifying from database:
   bag_id receipt_no  pluck_date month_start  year  month worker_id     full_name  bag_no  act_wt_kg  net_wt_kg  rate_kes_per_kg  pay_kes   centre_name clerk_name
0       1  330039762  2026-01-14  2026-01-01  2026      1    W-0042  Mary Wanjiru       1       12.3       11.8             15.0    177.0  NEW KIARUHIU       None
1       2  330039762  2026-01-14  2026-01-01  2026      1    W-0017    John Kamau       2       12.5       12.0             15.0    180.0  NEW KIARUHIU       None


C:\Users\mukil\AppData\Local\Temp\ipykernel_1872\1446364220.py:91: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM v_payments_due", conn)
